# Whisper STT

A refresher on **Whisper** — OpenAI's open-weight, multilingual *automatic speech recognition* (ASR) model. You hand it audio; it hands back text (and, optionally, timestamps, language ID, or a translation into English). It was trained on ~680k hours of weakly-supervised, web-scraped audio, which is why it's astonishingly robust to accents, background noise, and domain jargon straight out of the box — no fine-tuning, no per-speaker setup.

**Domain:** Speech & Audio  ·  **from study list**  ·  **runnable:** yes  ·  _front-end (mel) runs on CPU with no download; the model download is gated behind `RUN_WHISPER`_

## 1. What & Why

**What it is.** Whisper is a single sequence-to-sequence Transformer (encoder–decoder) that maps a log-mel spectrogram of audio to a sequence of text tokens. One model does several tasks, selected by special prompt tokens: **transcribe** (audio → same-language text), **translate** (audio → English text), **language identification**, and **voice-activity / timestamp** prediction. Sizes run `tiny`, `base`, `small`, `medium`, `large-v3` (plus the distilled `large-v3-turbo`), trading accuracy for speed and memory.

**The problem it solves.** Before Whisper, robust ASR meant either a paid cloud API (Google, AWS, Azure) or a brittle pipeline (Kaldi, DeepSpeech) that needed careful tuning, a language model, and clean audio to behave. Whisper gave the community a *single open-weights file* that transcribes ~100 languages, handles noisy real-world audio, punctuates and capitalizes, and runs anywhere from a laptop CPU to a GPU — for free.

**When to reach for it.** Batch transcription of podcasts, meetings, interviews, lectures; generating subtitles/captions; speech-to-text for a voice assistant where a second or two of latency is fine; any-language → English translation. It is the default, no-questions-asked starting point for offline ASR.

**When not to.** *Real-time / streaming* (Whisper is built around fixed **30-second** windows, so naïvely it's batch, not streaming — use `faster-whisper`/WhisperX or a streaming model for low latency). *Speaker diarization* ("who spoke when") — Whisper doesn't do it; pair it with `pyannote`. *Tiny-footprint edge devices* where even `tiny` is too heavy. And it **hallucinates** confident text on silence/non-speech, so it's risky where a wrong-but-fluent transcript is worse than none.

## 2. Mental Model

**Whisper is "GPT for audio": an encoder that *reads* a fixed 30-second picture of the sound, and a decoder that *writes* the transcript token by token, conditioned on prompt tokens that tell it what job to do.**

```
  raw audio            front-end                 ENCODER              DECODER (autoregressive)
 ┌──────────┐   resample 16 kHz mono   ┌──────────────────┐   ┌────────────────────────────┐
 │  any wav │──▶ pad/trim to 30 s   ──▶│ 2 conv + N self- │──▶│ cross-attends to audio,     │
 │  mp3 ... │   log-mel spectrogram    │ attention blocks │   │ emits text tokens given a   │
 └──────────┘     (80 × 3000)          │ -> audio states  │   │ prompt of special tokens    │
                                       └──────────────────┘   └────────────────────────────┘
                                                                         │
  <|startoftranscript|><|en|><|transcribe|><|notimestamps|>  ──────────▶ "Hello world."
        ^^ these prompt tokens choose language + task; the decoder is just predicting
           the next token, exactly like a language model.
```

Two ideas make everything click:

1. **Fixed 30-second chunks.** The encoder always sees exactly 30 s of audio as an `(80 mel bins × 3000 frames)` image — shorter clips are zero-padded, longer audio is sliced into 30 s windows and stitched. This is why "streaming Whisper" is awkward and why long files are processed chunk by chunk.
2. **The task is a *prompt*, not a *model*.** Transcribe vs. translate vs. language-ID are all the same weights; you just feed different special tokens at the start of the decoder. That's the GPT analogy taken literally — it's next-token prediction over a vocabulary that includes both words and control tokens.

## 3. Key Concepts

- **Log-mel spectrogram** — the model's actual input. Audio is resampled to **16 kHz mono**, then turned into an `80×3000` (or `128×3000` for `large-v3`) image via STFT → mel filterbank → log → normalize. Frames are 10 ms apart (hop 160) over 25 ms windows (n_fft 400). The front end is deterministic signal processing; the learning happens after it.
- **Encoder–decoder Transformer** — a convolutional stem downsamples the spectrogram, the **encoder** produces audio hidden states, and the **decoder** cross-attends to them while autoregressively generating text. Same architecture family as the original "Attention Is All You Need" model.
- **Special / prompt tokens** — `<|startoftranscript|>`, a language tag (`<|en|>`, `<|es|>`, …), a task tag (`<|transcribe|>` or `<|translate|>`), and `<|notimestamps|>` or timestamp tokens. These steer the single model across tasks and languages.
- **Model sizes** — `tiny`(39M) · `base`(74M) · `small`(244M) · `medium`(769M) · `large-v3`(1.55B), plus `large-v3-turbo` (a faster distilled decoder). Bigger = more accurate, slower, more VRAM. `.en` variants (e.g. `base.en`) are English-only and slightly better on English.
- **Decoding strategy** — greedy vs. beam search (`beam_size`), plus `temperature` fallback: if the output looks degenerate (high no-speech prob, low avg logprob, or excessive repetition), Whisper retries the chunk at a higher temperature. `condition_on_previous_text` feeds prior text as context (better coherence, but can propagate hallucinations).
- **Timestamps** — segment-level timestamps come from the timestamp tokens; **word-level** timestamps require cross-attention alignment (`word_timestamps=True` in the reference package, or WhisperX for tighter alignment).
- **Two ecosystems** — the reference **`openai-whisper`** package (`whisper.load_model`), and the **`transformers`** integration (`pipeline("automatic-speech-recognition", model="openai/whisper-...")`). For speed/long-form, **`faster-whisper`** (CTranslate2) is the production favorite — see the companion notebook.

## 4. Setup

Two interchangeable Python paths to the same weights:

```bash
# Reference implementation from OpenAI (CLI + simple API)
pip install -U openai-whisper        # also needs the ffmpeg binary on PATH

# Or via Hugging Face transformers (pipeline API, easy batching/devices)
pip install -U transformers torch
```

Whisper decodes audio with the **ffmpeg** binary, so install that from your OS package
manager (`brew install ffmpeg` / `apt-get install ffmpeg`) when feeding it file paths.
Models download from the hub on first use and cache locally (`~/.cache/whisper` or
`~/.cache/huggingface`): `tiny` ≈ 39 MB, `base` ≈ 145 MB, `large-v3` ≈ 3 GB.

To keep this notebook self-contained, the runnable cells below reproduce Whisper's
**front end** (the log-mel spectrogram) with only `numpy` + `torch` and a synthetic
signal — **no download, CPU-only**. The actual model call is shown but gated behind
`RUN_WHISPER` so the notebook always executes.

## 5. Worked Examples

### Example 1 — Synthesize 16 kHz audio (the only format Whisper accepts)

Whisper internally resamples everything to **mono, 16 kHz, float32**. We make a 4-second
rising chirp plus a little noise — no file, no download — and confirm its shape and
amplitude range, the contract every downstream cell relies on.

In [1]:
import os
import numpy as np
import torch

SR = 16000  # Whisper always operates at 16 kHz mono

# Synthesize 4 s of audio: a frequency sweep (chirp) 200 -> 2000 Hz plus faint noise.
dur = 4.0
t = np.linspace(0, dur, int(SR * dur), endpoint=False)
freq = np.linspace(200, 2000, t.size)                 # instantaneous frequency
phase = 2 * np.pi * np.cumsum(freq) / SR              # integrate freq -> phase
audio = 0.6 * np.sin(phase)
audio += 0.01 * np.random.default_rng(0).standard_normal(t.size)
audio = audio.astype(np.float32)

print(f"torch {torch.__version__}, numpy {np.__version__}")
print(f"audio: {audio.shape[0]} samples = {audio.shape[0]/SR:.1f}s @ {SR} Hz, dtype={audio.dtype}")
print(f"range: [{audio.min():.2f}, {audio.max():.2f}]  (Whisper wants mono float32 in [-1, 1])")

torch 2.12.1, numpy 2.5.0
audio: 64000 samples = 4.0s @ 16000 Hz, dtype=float32
range: [-0.64, 0.63]  (Whisper wants mono float32 in [-1, 1])


### Example 2 — Build Whisper's front end: the `(80, 3000)` log-mel spectrogram

This is the exact representation the encoder consumes. The recipe is fixed signal
processing — **no learned weights** — and reproducing it demystifies the model: pad/trim
the clip to exactly **30 s**, take an STFT (`n_fft=400`, `hop=160` → 10 ms frames),
project the power spectrum through **80 triangular mel filters**, take `log10`, and
normalize. The result is the `(80 mels × 3000 frames)` "image" Whisper-tiny expects.

In [2]:
N_FFT, HOP, N_MELS, CHUNK = 400, 160, 80, 30 * SR  # Whisper's fixed front-end params

def hz_to_mel(f): return 2595.0 * np.log10(1.0 + f / 700.0)
def mel_to_hz(m): return 700.0 * (10.0 ** (m / 2595.0) - 1.0)

def mel_filterbank(sr, n_fft, n_mels):
    """80 triangular filters spaced evenly on the mel scale (HTK convention)."""
    mels = np.linspace(hz_to_mel(0), hz_to_mel(sr / 2), n_mels + 2)
    bins = np.floor((n_fft + 1) * mel_to_hz(mels) / sr).astype(int)
    fb = np.zeros((n_mels, n_fft // 2 + 1), np.float32)
    for m in range(1, n_mels + 1):
        lo, ctr, hi = bins[m - 1], bins[m], bins[m + 1]
        for k in range(lo, ctr): fb[m - 1, k] = (k - lo) / max(ctr - lo, 1)
        for k in range(ctr, hi): fb[m - 1, k] = (hi - k) / max(hi - ctr, 1)
    return fb

# Whisper pads (or trims) every clip to exactly 30 s before the STFT.
padded = np.zeros(CHUNK, np.float32)
padded[:min(audio.size, CHUNK)] = audio[:CHUNK]

window = torch.hann_window(N_FFT)
stft = torch.stft(torch.from_numpy(padded), N_FFT, HOP, window=window, return_complex=True)
power = stft[..., :-1].abs().pow(2)                    # drop final frame -> (201, 3000)
mel = torch.from_numpy(mel_filterbank(SR, N_FFT, N_MELS)) @ power
logmel = torch.clamp(mel, min=1e-10).log10()
logmel = torch.maximum(logmel, logmel.max() - 8.0)     # dynamic-range floor (8 decades)
logmel = (logmel + 4.0) / 4.0                          # normalize, as Whisper does

print(f"raw 30s window  : {tuple(padded.shape)}  ({CHUNK // SR}s @ {SR} Hz)")
print(f"log-mel features: {tuple(logmel.shape)}  -> (n_mels, frames); Whisper-tiny wants (80, 3000)")
print(f"value range     : [{logmel.min():.2f}, {logmel.max():.2f}]")
# The 4 s chirp lives in the first ~400 frames; energy climbs through higher mel bins.
speech = logmel[:, :400]
print(f"loudest mel bin over the chirp: #{int(speech.mean(1).argmax())} of {N_MELS} "
      f"(energy migrates upward as the tone sweeps 200->2000 Hz)")

raw 30s window  : (480000,)  (30s @ 16000 Hz)
log-mel features: (80, 3000)  -> (n_mels, frames); Whisper-tiny wants (80, 3000)
value range     : [-0.09, 1.91]
loudest mel bin over the chirp: #37 of 80 (energy migrates upward as the tone sweeps 200->2000 Hz)


### Example 3 — The actual transcription call (gated behind `RUN_WHISPER`)

Once the model exists, transcription is one line: it takes a 16 kHz array (or a file
path) and returns text. The download is ~150 MB, so this is gated — set `RUN_WHISPER=1`
to run it for real. Either way the cell prints the canonical call shapes for both the
`transformers` pipeline and the reference `openai-whisper` package.

In [3]:
if os.getenv("RUN_WHISPER"):
    from transformers import pipeline
    asr = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")
    # The pipeline accepts a raw 16 kHz float32 array directly (or a file path / URL).
    result = asr({"array": audio, "sampling_rate": SR})
    print("transcription:", result["text"])  # a chirp -> Whisper will hallucinate; that's expected
else:
    print("Set RUN_WHISPER=1 to download openai/whisper-tiny (~150 MB) and transcribe.\n")
    print("transformers (Hugging Face):")
    print('    from transformers import pipeline')
    print('    asr = pipeline("automatic-speech-recognition", model="openai/whisper-base")')
    print('    asr("speech.mp3", return_timestamps=True)["text"]\n')
    print("reference openai-whisper package:")
    print('    import whisper')
    print('    model = whisper.load_model("base")        # tiny|base|small|medium|large-v3')
    print('    result = model.transcribe("speech.mp3", language="en")  # or task="translate"')
    print('    print(result["text"])')

Set RUN_WHISPER=1 to download openai/whisper-tiny (~150 MB) and transcribe.

transformers (Hugging Face):
    from transformers import pipeline
    asr = pipeline("automatic-speech-recognition", model="openai/whisper-base")
    asr("speech.mp3", return_timestamps=True)["text"]

reference openai-whisper package:
    import whisper
    model = whisper.load_model("base")        # tiny|base|small|medium|large-v3
    result = model.transcribe("speech.mp3", language="en")  # or task="translate"
    print(result["text"])


## 6. Gotchas & Pitfalls

- **Hallucination on silence/non-speech.** Whisper will confidently invent text on
  silence, music, or breathing — often "Thank you." or repeated phrases, and frequently
  YouTube-isms like "Subtitles by …" baked in from training data. Run VAD first
  (`faster-whisper` has `vad_filter=True`) and check `no_speech_prob`/`avg_logprob`.
- **Repetition loops.** The decoder can get stuck repeating a phrase. The built-in
  temperature fallback usually breaks it; `condition_on_previous_text=False` helps when
  a bad chunk poisons everything after it.
- **It's not streaming.** The 30-second window means naïve real-time use adds latency
  and chops words at boundaries. For live captions use a streaming wrapper or
  `faster-whisper` with manual chunking + overlap, not vanilla Whisper.
- **Wrong language auto-detected.** Language ID runs on only the **first 30 s**; if it
  opens with music or the wrong language, the whole file is misdetected. Pass
  `language="en"` explicitly when you know it.
- **Always feed 16 kHz mono.** Other sample rates/stereo get silently resampled, but if
  you bypass ffmpeg and pass a raw array at the wrong rate, output is garbage. Resample
  to 16 kHz mono yourself when in doubt (`ffmpeg -ar 16000 -ac 1`).
- **`fp16` warning on CPU.** Whisper defaults to fp16; on CPU it warns and falls back to
  fp32 (slower). Pass `fp16=False` to silence it. GPU memory scales with model size —
  `large-v3` needs ~10 GB VRAM.
- **Word timestamps ≠ free.** Segment timestamps are cheap; accurate *word*-level timing
  needs `word_timestamps=True` (extra compute) or WhisperX's forced alignment.
- **`large-v3` uses 128 mel bins, not 80.** If you hand-build the front end (as above) or
  swap models, match `n_mels` to the checkpoint or the shapes won't line up.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs vanilla Whisper |
|---|---|---|
| **openai-whisper (reference)** | Correctness baseline, research, the canonical API/CLI | Slowest runtime; PyTorch-only; batch, not streaming |
| **faster-whisper (CTranslate2)** | Production batch transcription, long files | 4–5× faster, far less memory, built-in VAD — but a separate library/format (see companion notebook) |
| **WhisperX** | Word-accurate timestamps + diarization | Adds alignment + `pyannote`; heavier setup, more deps |
| **whisper.cpp** | On-device / edge, no Python, quantized GGUF | C++ build; fewer knobs; great for laptops/phones |
| **NVIDIA NeMo / Parakeet, wav2vec2** | Streaming, ultra-low latency, custom fine-tuning | More setup; Whisper is more robust zero-shot across languages |
| **Cloud ASR (AWS Transcribe, Google, Deepgram, AssemblyAI)** | Managed scale, diarization, real-time SDKs, SLAs | Per-minute cost, data leaves your machine, vendor lock-in |

**Rule of thumb:** reach for Whisper first for any *offline, multilingual, accuracy-over-
latency* transcription — it's the strongest zero-shot open model. Switch to
`faster-whisper` the moment throughput or memory matters, to WhisperX when you need
precise word timing or "who said what," and to a streaming/cloud option when sub-second
latency is non-negotiable.

## 8. Resources

- **Reference implementation (GitHub)** — install, CLI, API, model card: https://github.com/openai/whisper
- **Paper — *Robust Speech Recognition via Large-Scale Weak Supervision*** — the why/how: https://arxiv.org/abs/2212.04356
- **Hugging Face Whisper docs** — `transformers` pipeline, long-form, fine-tuning: https://huggingface.co/docs/transformers/model_doc/whisper
- **faster-whisper** — the production-grade CTranslate2 reimplementation: https://github.com/SYSTRAN/faster-whisper
- **WhisperX** — word-level timestamps + speaker diarization: https://github.com/m-bain/whisperX